# Proceso de Notificación: Traslados de Salida BDUA (S4 vs R4)

Este notebook tiene como objetivo identificar, consolidar y preparar la información de los afiliados cuyo traslado de **Capresoca EPS** hacia otras entidades ha sido aprobado y procesado en la **Base de Datos Única de Afiliados (BDUA)**.

## Objetivo General
Clasificar y extraer la información de contacto (teléfono y correo electrónico) de los usuarios que se retiran de la EPS en un periodo específico (Mes/Año), generando un insumo estandarizado para las campañas de notificación masiva vía SMS y Email.

## Flujo de Datos
1.  **Carga de Fuentes BDUA:** Lectura de archivos maestros de traslados (S4) y sus respectivas respuestas de validación (R4).
2.  **Cruce de Integridad:** Identificación de registros aprobados mediante la comparación de estructuras S4 y R4.
3.  **Enriquecimiento de Contacto:** Cruce con el Maestro de Afiliados o bases de datos de contacto para obtener datos actualizados de localización.
4.  **Generación de Insumo:** Exportación a formato Excel con validación de estructura para herramientas de envío masivo.

## Alcance del Periodo
Este proceso es **idempotente por periodo**. Cada ejecución debe parametrizarse para el mes de corte correspondiente, asegurando que no se dupliquen notificaciones de meses anteriores.


# 1.0. Modulos

### 1. Explicación técnica del bloque
Este bloque inicializa el entorno de ejecución para el notebook de traslados. Realiza tres tareas críticas:
- **Importación de dependencias**: Carga librerías para manipulación vectorial (`pandas`, `numpy`), gestión de archivos (`pathlib`), validación de patrones de contacto (`re`) y gestión de memoria (`gc`).
- **Inyección del Path**: Implementa una lógica de búsqueda dinámica para localizar la raíz del repositorio. Esto permite que el notebook funcione sin importar en qué subcarpeta (`notebooks/Aseguramiento/`, etc.) se encuentre, facilitando el uso del módulo `config`.
- **Carga de Configuración**: Importa el archivo `config.py` del repositorio, el cual detecta automáticamente si el usuario es Yesid u otro desarrollador y ajusta las rutas de OneDrive en consecuencia.

### 2. Justificación de decisiones de optimización
- **Pathlib vs Os.path**: Se prioriza `pathlib` por ser más legible y manejar correctamente las barras inclinadas en sistemas Windows (común en el entorno Capresoca).
- **GC Collect**: Se invoca el recolector de basura tras la configuración inicial para asegurar que el espacio de direccionamiento de memoria esté limpio antes de cargar los DataFrames maestros (S4/R4), que suelen ser pesados.

### 3. Validaciones de integridad aplicadas
- **Validación de Importación**: Se implementa un bloque `try-except` para la carga de `config.py`. Si el archivo no existe o la ruta es incorrecta, el proceso se detiene inmediatamente (**Honestidad Brutal**) para evitar errores en cascada por rutas no definidas.
- **Detección de Raíz**: El bucle `while` garantiza que el entorno de Python reconozca los módulos locales del repositorio.

### 4. Listado de variables temporales eliminadas
- `project_root`: Eliminada tras inyectarse en `sys.path`.

### 5. Posibles advertencias
- Si se utiliza un entorno virtual (venv), asegúrese de que el kernel de Jupyter seleccionado en VS Code sea el correspondiente para evitar conflictos de versiones en `pandas` o `numpy`.

### 6. Validaciones sugeridas
- Ejecutar `config.info()` en la siguiente celda para confirmar que las rutas detectadas hacia `ONEDRIVE_BASE` y `BDUA_BASE` son correctas y accesibles.

In [ ]:
import os
import sys
import re
import gc
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
from datetime import date
from typing import List, Dict, Any, Optional

# 1. Configuración de rutas del proyecto y acceso a config.py
# Se busca de forma recursiva hacia arriba la raíz del proyecto que contiene 'config.py'
project_root: Path = Path.cwd()
while not (project_root / 'config.py').exists() and project_root != project_root.parent:
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

try:
    import config
    print(f"✅ Configuración cargada exitosamente desde: {project_root}")
except ImportError as e:
    print(f"❌ Error al importar config.py: {e}")
    # Detenemos la ejecución si la configuración base es crítica
    raise

# 2. Configuración de visualización de pandas para desarrollo
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# Limpieza de variables de inicialización de rutas
del project_root
gc.collect()

# 2 Rutas
### 1. Explicación técnica del bloque
Este bloque centraliza la gestión de rutas del proyecto eliminando la dependencia de nombres de usuario locales (`hardcoding`). 
- Utiliza la constante `config.CAPRESOCA_BASE` y `config.YESID_BASE`, las cuales resuelven internamente la ruta hasta OneDrive detectando el usuario del sistema operativo.
- Se implementa el uso de `f-strings` para la variable `FECHA_MAESTRO_SIE` y la ruta de salida, permitiendo que el cambio de nombre del archivo SIE y la carpeta del informe mensual sean tareas de un solo paso.
- Se definen variables para los periodos de proceso (`AÑO_PROCESO`, `MES_PROCESO`, `NUM_INFORME`) que automatizan la jerarquía de carpetas de salida.

### 2. Justificación de decisiones de optimización
- **Pathlib Joinery**: Se utiliza el operador `/` de `pathlib` para unir rutas. Esto es más seguro que concatenar cadenas, gestionando automáticamente los separadores de Windows.
- **Validación Preventiva**: Antes de cargar DataFrames pesados, el código verifica la existencia de los archivos. Esto ahorra CPU/RAM al fallar rápido si hay un error de ruta o sincronización.
- **Creación de Directorios**: `mkdir(parents=True)` asegura que la estructura de salida se cree automáticamente si es un nuevo mes de informe.

### 3. Validaciones de integridad aplicadas
- **Principio de Honestidad Brutal**: El bucle de validación recorre las rutas críticas e informa visualmente si alguna no es accesible antes de proceder.
- **Validación de tipos**: Se asegura que las rutas sean objetos `Path` para habilitar métodos de validación avanzados.

### 4. Listado de variables temporales eliminadas
- `rutas_criticas`: Diccionario temporal de validación.
- `nombre`, `ruta`, `estado`: Variables de iteración.

### 5. Posibles advertencias
- **Sincronización de OneDrive**: Si los archivos aparecen como "No encontrados", verifique que OneDrive esté en estado "Disponible en este dispositivo".
- **Estructura de Carpetas de Salida**: El script asume la nomenclatura `CTO 102.2026`. Si el contrato cambia, debe ajustarse en la definición de `R_SALIDA_BASE`.

### 6. Validaciones sugeridas
- Confirmar que el archivo `Reporte_Validación Archivos Maestro_...csv` no esté bloqueado por otra aplicación (Excel) al momento de la lectura.

In [ ]:
import gc
from pathlib import Path
from datetime import date

# ============================================================================
# PARAMETRIZACIÓN DEL PERIODO (Único punto de actualización mensual)
# ============================================================================
# Fecha para filtrar registros en BDUA y organizar carpetas
AÑO_PROCESO: int = 2026
MES_PROCESO: int = 2
# Número de informe para la ruta de salida (ej: "03" para el Informe #03)
NUM_INFORME: str = "03"

# Fecha específica del reporte Maestro SIE (se actualiza según el archivo disponible)
FECHA_MAESTRO_SIE: str = "2026_03_16"

# ============================================================================
# DEFINICIÓN DE RUTAS DINÁMICAS (Basadas en config.py)
# ============================================================================

# Rutas BDUA - Subsidiado (EPS025)
R_S4_SUBSIDIADO: Path = config.CAPRESOCA_BASE / "Procesos BDUA" / "Subsidiados" / "Procesos BDUA EPS" / "S4" / "S4_consolidado_total.txt"
R_R4_SUBSIDIADO: Path = config.CAPRESOCA_BASE / "Procesos BDUA" / "Subsidiados" / "Procesos BDUA EPS" / "R4" / "R4_consolidado_total.txt"

# Rutas BDUA - Contributivo (EPSC25)
R_S4_CONTRIBUTIVO: Path = config.CAPRESOCA_BASE / "Procesos BDUA" / "Contributivo" / "Procesos BDUA EPS" / "S4" / "S4_consolidado_total.txt"
R_R4_CONTRIBUTIVO: Path = config.CAPRESOCA_BASE / "Procesos BDUA" / "Contributivo" / "Procesos BDUA EPS" / "R4" / "R4_consolidado_total.txt"

# Ruta Maestro SIE (Contactos)
R_MAESTRO_SIE: Path = config.SIE_BASE / "Aseguramiento" / "ms_sie" / f"Reporte_Validación Archivos Maestro_{FECHA_MAESTRO_SIE}.csv"

# Ruta Municipios SIE (Contactos)
R_MUNICIPIOS_SIE: Path = config.SIE_BASE / "codificación de variables categóricas" / "Reporte_MUNICIPIOS_2025_05_14.csv"

# Ruta Constantes y Equivalencias EPS
R_CORREOS_EPS: Path = config.CONSTANTES_DIR / "CORREO DE OTRAS EPS.xlsx"
HOJA_CORREOS_EPS: str = "correos eps"

# Ruta de Salida (Directorio para el Excel de notificaciones)
# Se construye dinámicamente usando YESID_BASE detectado en config.py
R_SALIDA_BASE: Path = config.YESID_BASE / "informes" / str(AÑO_PROCESO) / f"CTO 102.{AÑO_PROCESO}" / f"CTO102.{AÑO_PROCESO} Informe  #{NUM_INFORME}" / "12 Actividad" / "Bases de datos notificaciones telefonicas" / "TRASLADOS A OTRAS EPS (SAT) - BDUA"

# Asegurar que el directorio de salida exista
R_SALIDA_BASE.mkdir(parents=True, exist_ok=True)

# ============================================================================
# VALIDACIÓN DE EXISTENCIA
# ============================================================================
rutas_criticas = {
    "S4 Subsidiado": R_S4_SUBSIDIADO,
    "R4 Subsidiado": R_R4_SUBSIDIADO,
    "Maestro SIE": R_MAESTRO_SIE,
    "Municipios SIE": R_MUNICIPIOS_SIE,
    "Excel Correos EPS": R_CORREOS_EPS,
    "Carpeta de Salida": R_SALIDA_BASE
}

print(f"{'Archivo/Ruta':<20} | {'Estado':<10}")
print("-" * 35)
for nombre, ruta in rutas_criticas.items():
    estado = "✅ Ok" if ruta.exists() else "❌ No encontrado"
    print(f"{nombre:<20} | {estado}")

# Limpieza de variables temporales
del rutas_criticas, nombre, ruta, estado
gc.collect()

# 3. Dataframes

### 1. Explicación técnica del bloque
Este bloque realiza la ingesta de datos de cuatro fuentes distintas, aplicando una lógica de consolidación para los procesos BDUA:
- **Consolidación S4/R4**: Se cargan por separado los archivos de los regímenes Subsidiado (`EPS025`) y Contributivo (`EPSC25`), inyectando una columna de origen antes de unificarlos con `pd.concat`.
- **Carga Estricta**: Se fuerza el parámetro `dtype=str` en todas las lecturas. Esto evita que Python interprete números de identificación como científicos o elimine ceros a la izquierda en códigos de EPS o teléfonos.
- **Fuentes de Apoyo**: Se integra el Maestro SIE para futuros cruces de contacto y el archivo de constantes para la identificación de las EPS receptoras.

### 2. Justificación de decisiones de optimización
- **dtype=str**: Aunque consume más memoria que tipos numéricos, es la única forma de garantizar el **Principio 1 (Calidad del Dato Supremo)** al tratar con documentos de identidad y códigos institucionales.
- **Uso de Concatenación**: Se prefiere `pd.concat` sobre `append` (depreciado) por ser más eficiente en memoria y tiempo de ejecución al reconstruir el índice una sola vez.
- **Estandarización de Codificación**: Se utiliza de manera uniforme `encoding='latin-1'` para asegurar la lectura correcta de caracteres especiales (tildes, eñes) comunes en las bases de datos locales y archivos planos de BDUA.

### 3. Validaciones de integridad aplicadas
- **Dimensionalidad**: Se compara el conteo de columnas entre los archivos de ambos regímenes antes de intentar la unión. Si las estructuras difieren, el proceso notifica el error en lugar de generar un DataFrame corrupto.
- **Manejo de Vacíos**: Se verifica la existencia de los DataFrames individuales para evitar errores al intentar concatenar objetos no iterables.

### 4. Listado de variables temporales eliminadas
- `df_s4_sub`, `df_s4_con`: Fragmentos originales de S4 cargados por régimen.
- `df_r4_sub`, `df_r4_con`: Fragmentos originales de R4 cargados por régimen.

### 5. Posibles advertencias
- **Delimitadores**: Se asume coma `,` para los archivos planos. Si los consolidados usan un separador distinto como `|` o `;`, la dimensionalidad fallará y se obtendrá una sola columna.
- **Estructura del Maestro SIE**: Se asume que el reporte de validación del SIE mantiene el formato CSV estándar. Cambios en las columnas del SIE podrían afectar los cruces posteriores.

### 6. Validaciones sugeridas
- Ejecutar `df_s4_total['REGIMEN'].value_counts()` para confirmar que la distribución de registros entre EPS025 y EPSC25 es la esperada y que ninguna categoría quedó en cero.
- Verificar con `df_sie.head()` que nombres, apellidos y teléfonos se visualicen correctamente con sus caracteres especiales gracias al `latin-1`.
- Validar que no existan valores `NaN` masivos en `df_correos_eps` tras la carga del Excel.

In [ ]:
# ============================================================================
# FUNCIONES DE CARGA Y CONSOLIDACIÓN
# ============================================================================

def cargar_y_etiquetar(ruta: Path, regimen: str, sep: str = ',') -> pd.DataFrame:
    """
    Carga un archivo plano como string, usa la primera fila como encabezado
    y añade la columna de REGIMEN.
    """
    if not ruta.exists():
        print(f"❌ Error: No se encontró el archivo en {ruta}")
        return pd.DataFrame()
    
    # Carga total como str para preservar integridad (Principio 1)
    # header=0 indica que la primera fila contiene los nombres de las columnas
    df = pd.read_csv(
        ruta, 
        sep=sep, 
        dtype=str, 
        encoding='latin-1', 
        engine='python',
        header=0
    )
    df['REGIMEN'] = regimen
    return df

# ============================================================================
# PROCESAMIENTO DE TRASLADOS (S4 y R4)
# ============================================================================

# Consolidación S4
print("Cargando archivos S4...")
df_s4_sub = cargar_y_etiquetar(R_S4_SUBSIDIADO, 'EPS025')
df_s4_con = cargar_y_etiquetar(R_S4_CONTRIBUTIVO, 'EPSC25')

# Validación de dimensionalidad antes del merge (Principio 1)
if not df_s4_sub.empty and not df_s4_con.empty:
    if len(df_s4_sub.columns) == len(df_s4_con.columns):
        df_s4_total = pd.concat([df_s4_sub, df_s4_con], ignore_index=True)
        print(f"✅ S4 Consolidado: {len(df_s4_total):,} registros.")
    else:
        print(f"⚠️ Error: Columnas no coinciden. Sub: {len(df_s4_sub.columns)}, Con: {len(df_s4_con.columns)}")
        df_s4_total = pd.DataFrame()
else:
    df_s4_total = pd.concat([df_s4_sub, df_s4_con], ignore_index=True)

# Consolidación R4
print("\nCargando archivos R4...")
df_r4_sub = cargar_y_etiquetar(R_R4_SUBSIDIADO, 'EPS025')
df_r4_con = cargar_y_etiquetar(R_R4_CONTRIBUTIVO, 'EPSC25')

if not df_r4_sub.empty and not df_r4_con.empty:
    if len(df_r4_sub.columns) == len(df_r4_con.columns):
        df_r4_total = pd.concat([df_r4_sub, df_r4_con], ignore_index=True)
        print(f"✅ R4 Consolidado: {len(df_r4_total):,} registros.")
    else:
        print(f"⚠️ Error: Columnas no coinciden. Sub: {len(df_r4_sub.columns)}, Con: {len(df_r4_con.columns)}")
        df_r4_total = pd.DataFrame()
else:
    df_r4_total = pd.concat([df_r4_sub, df_r4_con], ignore_index=True)

# ============================================================================
# CARGA DE FUENTES DE APOYO (SIE y CORREOS EPS)
# ============================================================================

print("\nCargando Maestro SIE y Directorio de EPS...")
# Maestro SIE (Separador punto y coma, primera fila como encabezado)
df_sie = pd.read_csv(R_MAESTRO_SIE, sep=';', dtype=str, encoding='latin-1', header=0)
df_municipios_sie = pd.read_csv(R_MUNICIPIOS_SIE, sep=';', dtype=str, encoding='latin-1', header=0)

# Directorio de correos EPS (Excel utiliza por defecto la primera fila como header)
df_correos_eps = pd.read_excel(R_CORREOS_EPS, sheet_name=HOJA_CORREOS_EPS, dtype=str)

# ============================================================================
# LIMPIEZA DE MEMORIA
# ============================================================================
# Liberar variables temporales pesadas (Principio 3)
del df_s4_sub, df_s4_con, df_r4_sub, df_r4_con
gc.collect()

# 4. Limpieza de datos

## Unificación S4 y R4
### 1. Explicación técnica del bloque
Este bloque resuelve la asimetría estructural entre las fuentes de traslados:
- **Remoción de Columna**: Se elimina `ID_NOM_REGIS_ARCHIVO` de `df_r4_total` para que coincida con la estructura de `df_s4_total`.
- **Alineación de Columnas**: Se fuerza a que `df_s4_total` adopte el mismo orden de columnas que `df_r4_total` antes de la concatenación, evitando que `pandas` genere columnas duplicadas o valores `NaN` por desorden.
- **Consolidación Final**: Se crea el objeto `df_traslados_consolidado` que unifica ambos tipos de archivo, añadiendo la etiqueta `TIPO_ARCHIVO` para distinguir si el registro proviene de un envío (S4) o una respuesta (R4).

### 2. Justificación de decisiones de optimización
- **Reordenamiento Explícito**: En lugar de confiar en que las columnas están en la misma posición, se alinean programáticamente. Esto es vital para garantizar la **Calidad del Dato** al usar archivos `.txt` que a veces varían en su generación.
- **Etiquetado Preventivo**: Al unificar, es mejor añadir una columna de origen (`TIPO_ARCHIVO`) que simplemente apilar los datos, ya que permite realizar auditorías rápidas sobre qué registros son aprobaciones directas y cuáles son solicitudes.

### 3. Validaciones de integridad aplicadas
- **Dimensionalidad**: Se compara el conteo de columnas antes de ejecutar el `concat`.
- **Validación de Volumen**: Se realiza una suma aritmética de las filas de los dataframes originales contra el resultado final para confirmar que no hubo pérdida de información durante la operación.
- **Idempotencia**: El uso de validaciones previas permite ejecutar la celda varias veces sin riesgo de errores por columnas inexistentes.

### 4. Listado de variables temporales eliminadas
- `df_r4_total`: DataFrame parcial de respuestas.
- `df_s4_total`: DataFrame parcial de salidas.
- `total_esperado`: Variable de control de registros.

### 5. Posibles advertencias
- **Duplicados**: Si un mismo afiliado aparece en S4 y en R4 para el mismo proceso, tendrá dos filas en el consolidado. Esto se depurará en la siguiente fase de filtrado por periodo.

### 6. Validaciones sugeridas
- Ejecutar `df_traslados_consolidado.groupby(['REGIMEN', 'TIPO_ARCHIVO']).size()` para obtener una matriz de control y asegurar que los 4 universos (S4/R4 x Sub/Con) están presentes.

In [ ]:
# ============================================================================
# 4.1. LIMPIEZA Y UNIFICACIÓN (S4 + R4)
# ============================================================================

# 1. Eliminar columna excedente en R4 para igualar estructuras (Principio 1)
# Se usa 'errors=ignore' para que el bloque sea idempotente
if 'ID_NOM_REGIS_ARCHIVO' in df_r4_total.columns:
    df_r4_total = df_r4_total.drop(columns=['ID_NOM_REGIS_ARCHIVO'])

# 2. Validación de dimensionalidad antes de la unión técnica
if len(df_r4_total.columns) == len(df_s4_total.columns):
    # Asegurar que el orden de las columnas sea idéntico para evitar desalineación
    df_s4_total = df_s4_total[df_r4_total.columns]
    
    # Unificar S4 y R4 en un solo repositorio de traslados
    # Añadimos una columna de 'ORIGEN_ARCHIVO' para no perder la trazabilidad
    df_r4_total['TIPO_ARCHIVO'] = 'R4'
    df_s4_total['TIPO_ARCHIVO'] = 'S4'
    
    df_traslados_consolidado = pd.concat([df_r4_total, df_s4_total], ignore_index=True)
    
    # 3. Validación de integridad: No se deben perder registros en la unión
    total_esperado = len(df_r4_total) + len(df_s4_total)
    if len(df_traslados_consolidado) == total_esperado:
        print(f"✅ Unificación exitosa: {len(df_traslados_consolidado):,} registros consolidados.")
    else:
        print(f"⚠️ Alerta de integridad: Se esperaban {total_esperado} registros pero hay {len(df_traslados_consolidado)}.")
else:
    print(f"❌ Error crítico: Las estructuras no coinciden. R4: {len(df_r4_total.columns)} cols, S4: {len(df_s4_total.columns)} cols.")
    df_traslados_consolidado = pd.DataFrame()

# ============================================================================
# LIMPIEZA DE MEMORIA
# ============================================================================
# Eliminamos los dataframes parciales ya que ahora tenemos el consolidado
del df_r4_total, df_s4_total
gc.collect()

## Filtra periodo
### 1. Explicación técnica del bloque
Este bloque realiza el filtrado cronológico del universo consolidado de traslados:
- **Conversión Vectorizada**: Se utiliza `pd.to_datetime` con un formato explícito (`%d/%m/%Y`) para garantizar que la interpretación de día y mes sea la correcta para el estándar BDUA.
- **Manejo de Errores**: Se aplica `errors='coerce'` para identificar registros con fechas malformadas en el origen, los cuales serán excluidos automáticamente por el filtro al convertirse en `NaT`.
- **Persistencia de Tipos**: El filtrado se realiza mediante una serie temporal auxiliar, preservando la columna original `FECHA_PROCESO` como `str` para evitar alteraciones en el formato visual de los reportes finales.

### 2. Justificación de decisiones de optimización
- **Filtrado por Máscara**: Se evita el uso de bucles o funciones `.apply()`. La comparación booleana sobre los atributos `.dt.month` y `.dt.year` es una operación vectorizada de alta velocidad en CPU.
- **Uso de `.copy()`**: Al aplicar el filtro, se genera una copia explícita del DataFrame para evitar el aviso `SettingWithCopyWarning` en transformaciones futuras y liberar el enlace de memoria con el DataFrame original más grande.

### 3. Validaciones de integridad aplicadas
- **Check de Vacíos**: El bloque valida si hay datos antes de iniciar para evitar errores de tipo `NoneType`.
- **Auditoría de Descarte**: Se calcula y muestra la cantidad de registros excluidos. Esto permite al usuario confirmar si el volumen descartado es razonable o si existe un error en los parámetros `MES_PROCESO` / `AÑO_PROCESO`.
- **Validación de Categorías**: Se imprime la distribución de fechas resultantes para confirmar que efectivamente todas pertenecen al mes y año solicitado (confirmando la lógica de 1 a 4 fechas por mes).

### 4. Listado de variables temporales eliminadas
- `fechas_temp`: Serie temporal de conversión.
- `mascara_periodo`: Array booleano de filtrado.
- `registros_iniciales`, `registros_finales`, `registros_eliminados`: Contadores de control.
- `distribucion_fechas_total`: Objeto de auditoría de fechas.

### 5. Posibles advertencias
- **Fechas Ambiguas**: Si el archivo de origen tuviera fechas en formato americano (mm/dd/yyyy), la conversión fallaría o daría resultados erróneos. Se ha forzado el formato latino para mitigar esto.
- **Volumen de Datos**: Si el DataFrame consolidado original es masivamente grande (millones de filas), la conversión a datetime puede tener un pico de uso de RAM momentáneo.

### 6. Validaciones sugeridas
- Ejecutar `df_traslados_consolidado['FECHA_PROCESO'].unique()` para visualizar rápidamente que no existan años o meses intrusos.
- Verificar que el total de registros coincida aproximadamente con las expectativas del área de aseguramiento para el mes de proceso.

In [ ]:
# ============================================================================
# 4.2. FILTRADO POR PERIODO CRÍTICO (AÑO/MES)
# ============================================================================

# 1. Validación de pre-condición: Verificar que el DataFrame no esté vacío
if df_traslados_consolidado.empty:
    print("❌ Error: El DataFrame consolidado está vacío. No se puede proceder con el filtrado.")
else:
    # Registros iniciales para validación de integridad (Principio 1)
    registros_iniciales: int = len(df_traslados_consolidado)
    
    # 2. Transformación temporal a datetime para filtrado preciso
    # Se usa una serie temporal para no alterar la columna original de tipo str
    print(f"Filtrando registros para el periodo: {MES_PROCESO:02d}/{AÑO_PROCESO}...")
    
    # Convertimos a datetime manejando posibles errores (errors='coerce')
    fechas_temp: pd.Series = pd.to_datetime(
        df_traslados_consolidado['FECHA_PROCESO'], 
        format='%d/%m/%Y', 
        errors='coerce'
    )
    
    # 3. Aplicación del filtro vectorizado (Optimización Eficiente)
    mascara_periodo = (fechas_temp.dt.month == MES_PROCESO) & (fechas_temp.dt.year == AÑO_PROCESO)
    
    # Guardamos la distribución de fechas ANTES de sobreescribir (para auditoría)
    distribucion_fechas_total = df_traslados_consolidado['FECHA_PROCESO'].value_counts().sort_index()
    
    # Aplicar el filtro
    df_traslados_consolidado = df_traslados_consolidado[mascara_periodo].copy()
    
    # 4. Estadísticas de Proceso e Integridad
    registros_finales: int = len(df_traslados_consolidado)
    registros_eliminados: int = registros_iniciales - registros_finales
    
    print("-" * 50)
    print(f"📊 RESUMEN DE FILTRADO")
    print("-" * 50)
    print(f"Registros antes del filtro: {registros_iniciales:,}")
    print(f"Registros después del filtro: {registros_finales:,}")
    print(f"Registros excluidos (otros periodos): {registros_eliminados:,}")
    
    if registros_finales > 0:
        print(f"\n✅ Distribución de fechas en el periodo seleccionado:")
        print(df_traslados_consolidado['FECHA_PROCESO'].value_counts().sort_index())
    else:
        print(f"⚠️ ADVERTENCIA: No se encontraron registros para {MES_PROCESO}/{AÑO_PROCESO}.")
        print("\nFechas disponibles en la fuente original (Top 5):")
        print(distribucion_fechas_total.head(5))

    # ============================================================================
    # LIMPIEZA DE MEMORIA
    # ============================================================================
    del fechas_temp, mascara_periodo, registros_iniciales, registros_finales, registros_eliminados, distribucion_fechas_total
    gc.collect()

## Filtrar salidas Aprovadas
### 1. Explicación técnica del bloque
Este bloque filtra el universo de traslados para obtener la base definitiva de notificaciones:
- **Criterio de Aprobación**: Se seleccionan exclusivamente los registros donde la columna `RESPUESTA` es igual a `"1"`. Esto identifica los movimientos que fueron aceptados por la BDUA.
- **Garantía de Unicidad**: Se aplica una limpieza sobre la columna `AFL_ID` (Identificador Único del Afiliado). Esto es fundamental para evitar el envío de múltiples notificaciones (SMS/Email) a la misma persona si esta aparece reportada en más de un archivo del mismo periodo.

### 2. Justificación de decisiones de optimización
- **Vectorización de Filtro**: La selección de aprobados se realiza mediante indexación booleana de alta velocidad.
- **Drop Duplicates Eficiente**: En lugar de agrupar (`groupby`), se utiliza `drop_duplicates` que es una operación optimizada en C para eliminar redundancias basándose en una clave específica sin la carga computacional de una agregación.
- **Keep='first'**: Se conserva la primera ocurrencia. Dado que el objetivo es la notificación, cualquier registro válido del periodo es suficiente para contactar al afiliado.

### 3. Validaciones de integridad aplicadas
- **Control de Flujo**: Se verifica que el DataFrame no esté vacío antes de operar.
- **Contabilidad de Descarte**: El código calcula explícitamente cuántos registros se pierden por no ser aprobados y cuántos por ser duplicados. Esto permite detectar si el filtro de "RESPUESTA" fue demasiado agresivo o si hubo un error en la carga previa.
- **Validación de Identificador**: Al limpiar por `AFL_ID`, nos aseguramos de que el resultado sea una lista de personas físicas únicas.

### 4. Listado de variables temporales eliminadas
- `regs_pre_aprobados`, `regs_post_aprobados`: Contadores de registros.
- `conteo_duplicados`: Cantidad de IDs repetidos detectados.
- `regs_finales`: Conteo final de la base.

### 5. Posibles advertencias
- **Pérdida de Información**: Si un afiliado tiene un traslado S4 aprobado y un R4 aprobado en el mismo mes, solo se conservará uno de los registros. Esto es correcto para notificaciones, pero si se requiere trazabilidad de *todos* los movimientos, se debe revisar la lógica de negocio.

### 6. Validaciones sugeridas
- Ejecutar `df_traslados_consolidado['AFL_ID'].nunique() == len(df_traslados_consolidado)` para confirmar matemáticamente que no quedan duplicados.
- Revisar una muestra con `df_traslados_consolidado.sample(5)` para asegurar que la columna `RESPUESTA` sea efectivamente "1" en todos los casos.

In [ ]:
# ============================================================================
# 4.3. FILTRADO DE APROBADOS Y UNICIDAD (AFL_ID)
# ============================================================================

# 1. Validación de pre-condición
if df_traslados_consolidado.empty:
    print("❌ Error: No hay datos para procesar en df_traslados_consolidado.")
else:
    # Registros antes del filtro de aprobación
    regs_pre_aprobados: int = len(df_traslados_consolidado)
    
    # 2. Filtrar solo traslados aprobados (RESPUESTA == '1')
    # Se utiliza comparación de strings ya que cargamos todo como 'str'
    df_traslados_consolidado = df_traslados_consolidado[
        df_traslados_consolidado['RESPUESTA'] == '1'
    ].copy()
    
    regs_post_aprobados: int = len(df_traslados_consolidado)
    
    # 3. Validación y eliminación de duplicados por AFL_ID (Principio 1 y 2)
    # Identificamos cuántos AFL_ID están duplicados antes de limpiar
    conteo_duplicados: int = df_traslados_consolidado.duplicated(subset=['AFL_ID']).sum()
    
    # Mantenemos el primer registro encontrado (habitualmente el más reciente o el de R4 
    # si el dataframe fue ordenado previamente, pero asegura un único registro por afiliado)
    df_traslados_consolidado = df_traslados_consolidado.drop_duplicates(
        subset=['AFL_ID'], 
        keep='first'
    )
    
    regs_finales: int = len(df_traslados_consolidado)

    # 4. Reporte Técnico de Integridad
    print("-" * 50)
    print(f"✅ PROCESO DE DEPURACIÓN DE APROBADOS")
    print("-" * 50)
    print(f"Registros iniciales (Periodo):      {regs_pre_aprobados:,}")
    print(f"Registros aprobados (Resp='1'):     {regs_post_aprobados:,}")
    print(f"Registros descartados (No aprob.):  {regs_pre_aprobados - regs_post_aprobados:,}")
    print(f"Afiliados duplicados eliminados:    {conteo_duplicados:,}")
    print(f"TOTAL AFILIADOS ÚNICOS A NOTIFICAR: {regs_finales:,}")
    print("-" * 50)

    # ============================================================================
    # LIMPIEZA DE MEMORIA
    # ============================================================================
    del regs_pre_aprobados, regs_post_aprobados, conteo_duplicados, regs_finales
    gc.collect()

# 5. Consolidar información

## 5.1. Nombre EPS

### 1. Explicación técnica del bloque
Este bloque resuelve la relación entre los códigos técnicos de las EPS (`ENT_ID_SOLICITANTE`) y sus nombres comerciales para el reporte final:
- **Normalización del Maestro**: Se eliminan duplicados en `df_correos_eps` basados en la cadena de códigos. 
- **Lógica de Expansión (Explode)**: Dado que el maestro agrupa códigos separados por guiones (ej. `ESS062-ESSC62`), se utiliza `.str.split('-')` seguido de `.explode()` para convertir esa lista en filas individuales. Esto permite que un `merge` sencillo localice cualquier código contenido en la cadena.
- **Enriquecimiento**: Se realiza un `left join` para traer el nombre de la `ENTIDAD` basándose en la coincidencia exacta del código.

### 2. Justificación de decisiones de optimización
- **Uso de Explode**: Es la forma más eficiente y legible de manejar relaciones "uno a muchos" codificadas en una sola celda de texto. Evita el uso de expresiones regulares complejas dentro del merge.
- **Limpieza de Strings**: Se aplica `.str.strip()` en ambos lados de la relación de cruce para prevenir fallos por espacios invisibles o tabulaciones, garantizando la **Calidad del Dato**.

### 3. Validaciones de integridad aplicadas
- **Check de Dimensionalidad**: Se compara el conteo de filas antes y después del cruce. Si el número de filas aumenta, el sistema alerta sobre una posible duplicidad en el maestro de correos (Principio 2: Reproducibilidad).
- **Identificación de Huérfanos**: Se contabilizan y listan los códigos de EPS que no existen en el Excel de correos. Esto permite al usuario actualizar el archivo maestro si aparecen nuevas EPS en el mercado.

### 4. Listado de variables temporales eliminadas
- `df_mapping`: DataFrame auxiliar de equivalencias expandido.
- `regs_antes_merge`, `regs_despues_merge`: Contadores de control.
- `eps_no_encontradas`: Variable de auditoría de calidad.

### 5. Posibles advertencias
- **Códigos Inexistentes**: Si el reporte muestra códigos en la sección de "⚠️ Códigos de EPS detectados sin nombre", los registros resultantes tendrán un valor `NaN` en el nombre de la EPS, lo cual afectará la estética del reporte final si no se corrige el Excel de entrada.

### 6. Validaciones sugeridas
- Ejecutar `df_traslados_consolidado[['ENT_ID_SOLICITANTE', 'NOMBRE_EPS_DESTINO']].drop_duplicates()` para verificar visualmente que la asignación de nombres es coherente (ej: EPS037 -> Nueva EPS).

In [ ]:
# ============================================================================
# 4.3. ENRIQUECIMIENTO: NOMBRE DE EPS DESTINO
# ============================================================================

# 1. Preparación del Maestro de Equivalencias (df_correos_eps)
# Eliminamos duplicados por la cadena de códigos para tener una relación única de Nombre-Códigos
df_mapping: pd.DataFrame = df_correos_eps[['ENTIDAD', 'COD ENTIDAD']].drop_duplicates(subset=['COD ENTIDAD']).copy()

# 2. Expansión de códigos (Explode)
# Como un mismo nombre de EPS puede tener varios códigos (EPS037-EPS041...), 
# los separamos y creamos una fila por cada código individual para el cruce.
df_mapping['COD_INDIVIDUAL'] = df_mapping['COD ENTIDAD'].str.split('-')
df_mapping = df_mapping.explode('COD_INDIVIDUAL')

# Limpieza de espacios en blanco accidentales para asegurar el cruce
df_mapping['COD_INDIVIDUAL'] = df_mapping['COD_INDIVIDUAL'].str.strip()
df_traslados_consolidado['ENT_ID_SOLICITANTE'] = df_traslados_consolidado['ENT_ID_SOLICITANTE'].str.strip()

# 3. Operación de Cruce (Merge)
# Registros antes del cruce para validación de integridad
regs_antes_merge: int = len(df_traslados_consolidado)

df_traslados_consolidado = df_traslados_consolidado.merge(
    df_mapping[['ENTIDAD', 'COD_INDIVIDUAL']], 
    left_on='ENT_ID_SOLICITANTE', 
    right_on='COD_INDIVIDUAL', 
    how='left'
)

# 4. Limpieza post-merge
# Renombramos la columna para mayor claridad y eliminamos la columna técnica de cruce
df_traslados_consolidado = df_traslados_consolidado.rename(columns={'ENTIDAD': 'NOMBRE_EPS_DESTINO'})
if 'COD_INDIVIDUAL' in df_traslados_consolidado.columns:
    df_traslados_consolidado = df_traslados_consolidado.drop(columns=['COD_INDIVIDUAL'])

# 5. Validaciones de Integridad (Principio 1)
regs_despues_merge: int = len(df_traslados_consolidado)

# Verificamos si hubo registros que no encontraron coincidencia
eps_no_encontradas: int = df_traslados_consolidado['NOMBRE_EPS_DESTINO'].isna().sum()

print("-" * 60)
print(f"📊 REPORTE DE ENRIQUECIMIENTO DE ENTIDADES")
print("-" * 60)
print(f"Registros procesados:              {regs_antes_merge:,}")
print(f"Cruce exitoso (con nombre):       {regs_antes_merge - eps_no_encontradas:,}")
print(f"EPS sin nombre (no en maestro):   {eps_no_encontradas:,}")

if eps_no_encontradas > 0:
    print(f"\n⚠️ Códigos de EPS detectados sin nombre en el maestro:")
    print(df_traslados_consolidado[df_traslados_consolidado['NOMBRE_EPS_DESTINO'].isna()]['ENT_ID_SOLICITANTE'].unique())

# Validación de dimensionalidad: No deben duplicarse filas en un Left Join si el mapping es correcto
if regs_antes_merge != regs_despues_merge:
    print(f"⚠️ ALERTA: La cantidad de registros cambió de {regs_antes_merge} a {regs_despues_merge}. "
          "Posible duplicidad en el archivo de correos.")

# ============================================================================
# LIMPIEZA DE MEMORIA
# ============================================================================
del df_mapping, regs_antes_merge, regs_despues_merge, eps_no_encontradas, df_correos_eps
gc.collect()

## 5.2. SIE [Apellidos, Nombres, Municipio, Telefonos, Correos] 

### 1. Explicación técnica del bloque
Este proceso integra la información de contacto personal desde la base de datos **SIE** hacia el consolidado de traslados:
- **Construcción de Llaves Compuestas**: Debido a que el número de identificación puede repetirse entre distintos tipos de documento (ej. RC y TI con el mismo número), se crea una `KEY` que concatena ambos valores para garantizar un cruce unívoco.
- **Normalización**: Se fuerza la conversión a `str` y se eliminan espacios en blanco (`strip`) para evitar fallos de cruce por formatos invisibles.
- **Enriquecimiento**: Se traen 8 campos críticos: nombres completos, apellidos, tres opciones de teléfono y el correo electrónico.

### 2. Justificación de decisiones de optimización
- **Pre-limpieza del SIE**: Antes del cruce, se eliminan duplicados del `df_sie` basándose en la llave documental. Esto es vital para evitar que una fila en el consolidado se multiplique si el usuario tiene varias entradas en el SIE (Principio 2).
- **Merge Selectivo**: No se une todo el `df_sie`, sino únicamente las columnas necesarias y la llave, optimizando el consumo de memoria RAM.

### 3. Validaciones de integridad aplicadas
- **Cálculo de Cobertura**: El reporte indica qué porcentaje de los afiliados que se trasladan están correctamente registrados en el SIE. Un porcentaje bajo podría indicar que el SIE no está actualizado con las últimas novedades de la BDUA.
- **Validación de Unicidad**: Se compara el número de registros antes y después. En un `left join`, si el conteo aumenta, significa que la llave de cruce no era única en la tabla de la derecha (SIE), lo cual dispararía una alerta técnica.

### 4. Listado de variables temporales eliminadas
- `df_sie_contactos`: Subconjunto optimizado del SIE.
- `KEY_TRASLADO` / `KEY_SIE`: Identificadores temporales de cruce.
- `regs_antes_contacto`, `regs_despues_contacto`: Contadores de auditoría.

### 5. Posibles advertencias
- **Datos Faltantes**: Los registros "No encontrados" quedarán con valores `NaN`. Para estos casos, se recomienda en la fase de salida utilizar fuentes alternativas o marcar como "Sin Información de Contacto" para el proceso de notificación masiva.

### 6. Validaciones sugeridas
- Ejecutar `df_traslados_consolidado[['celular', 'correo_electronico']].isna().mean() * 100` para conocer el porcentaje de "huecos" de información que hay para las notificaciones.

In [ ]:
# ============================================================================
# 5.1. ENRIQUECIMIENTO: DATOS DE CONTACTO (SIE)
# ============================================================================

# 1. Preparación de la llave de cruce en df_sie
# Creamos una llave única combinando Tipo y Número para asegurar precisión
df_sie_contactos = df_sie[[
    'tipo_documento', 'numero_identificacion', 
    'primer_apellido', 'segundo_apellido', 'primer_nombre', 'segundo_nombre', 'municipio', 
    'celular', 'telefono_1', 'telefono_2', 'correo_electronico'
]].copy()

# Normalización de la llave en SIE (eliminar espacios y asegurar strings)
df_sie_contactos['KEY_SIE'] = (
    df_sie_contactos['tipo_documento'].astype(str).str.strip() + 
    df_sie_contactos['numero_identificacion'].astype(str).str.strip()
)

# Eliminar duplicados en el SIE para evitar explosión de registros en el merge
# Mantenemos el último registro asumiendo que es el más actualizado en el sistema
df_sie_contactos = df_sie_contactos.drop_duplicates(subset=['KEY_SIE'], keep='last')

# 2. Preparación de la llave en df_traslados_consolidado
df_traslados_consolidado['KEY_TRASLADO'] = (
    df_traslados_consolidado['TPS_IDN_ID'].astype(str).str.strip() + 
    df_traslados_consolidado['HST_IDN_NUMERO_IDENTIFICACION'].astype(str).str.strip()
)

# 3. Operación de Cruce (Left Join)
regs_antes_contacto = len(df_traslados_consolidado)

df_traslados_consolidado = df_traslados_consolidado.merge(
    df_sie_contactos.drop(columns=['tipo_documento', 'numero_identificacion']), 
    left_on='KEY_TRASLADO', 
    right_on='KEY_SIE', 
    how='left'
)

# 4. Limpieza técnica
# Eliminamos las columnas auxiliares de cruce
cols_a_eliminar = ['KEY_TRASLADO', 'KEY_SIE']
df_traslados_consolidado = df_traslados_consolidado.drop(columns=[c for c in cols_a_eliminar if c in df_traslados_consolidado.columns])

# 5. Reporte de Calidad de Datos
regs_despues_contacto = len(df_traslados_consolidado)
hallados = df_traslados_consolidado['primer_nombre'].notna().sum()
no_hallados = df_traslados_consolidado['primer_nombre'].isna().sum()

print("-" * 60)
print(f"📊 REPORTE DE CRUCE CON BASE DE DATOS SIE")
print("-" * 60)
print(f"Total registros a enriquecer:      {regs_antes_contacto:,}")
print(f"Usuarios encontrados en SIE:       {hallados:,} ({(hallados/regs_antes_contacto)*100:.1f}%)")
print(f"Usuarios NO encontrados:           {no_hallados:,}")

if regs_antes_contacto != regs_despues_contacto:
    print(f"⚠️ ALERTA: Se generaron duplicados ({regs_despues_contacto - regs_antes_contacto}). "
          "Revise la integridad de las llaves en el SIE.")

# ============================================================================
# LIMPIEZA DE MEMORIA
# ============================================================================
del df_sie_contactos, regs_antes_contacto, regs_despues_contacto, hallados, no_hallados, df_sie
gc.collect()

### 5.2.1 Codigo municipios

In [ ]:
import gc
import pandas as pd

# ============================================================================
# 5.2. ENRIQUECIMIENTO: CÓDIGO DANE DE MUNICIPIO
# ============================================================================

# 0. Idempotencia y protección del estado del notebook
# Si la celda se re-ejecuta, eliminamos estructuras residuales para evitar errores
for col in ['Codigo_Municipio', 'KEY_MUN_JOIN']:
    if col in df_traslados_consolidado.columns:
        df_traslados_consolidado.drop(columns=[col], inplace=True)

# 1. Preparación de la tabla de mapeo (catálogo de municipios)
# Seleccionamos solo lo necesario y renombramos antes del cruce para evitar sufijos (_x, _y)
df_mun_map = df_municipios_sie[['descripcion', 'municipio']].rename(
    columns={'municipio': 'Codigo_Municipio'}
)

# Normalización estricta de la llave de cruce (texto)
# Convertimos a mayúsculas y eliminamos espacios en los extremos para mitigar errores de tipeo
df_mun_map['KEY_MUN_JOIN'] = df_mun_map['descripcion'].astype(str).str.strip().str.upper()

# Protección de granularidad: garantizar que el catálogo tenga llaves únicas
df_mun_map = df_mun_map.drop_duplicates(subset=['KEY_MUN_JOIN'], keep='first')

# 2. Preparación de la llave en la tabla transaccional (consolidado)
# Se asume que la columna de texto de municipio proveniente del cruce anterior se llama 'municipio'
# Reemplazamos NaN con string vacío para evitar que float(NaN) afecte la vectorización de strings
df_traslados_consolidado['KEY_MUN_JOIN'] = (
    df_traslados_consolidado['municipio']
    .fillna('')
    .astype(str)
    .str.strip()
    .str.upper()
)

# 3. Operación de Cruce (Left Join)
regs_antes_dane = len(df_traslados_consolidado)

df_traslados_consolidado = df_traslados_consolidado.merge(
    df_mun_map[['KEY_MUN_JOIN', 'Codigo_Municipio']],
    on='KEY_MUN_JOIN',
    how='left'
)

# 4. Limpieza de columnas temporales en el consolidado
df_traslados_consolidado.drop(columns=['KEY_MUN_JOIN'], inplace=True)

# 5. Reporte de Calidad de Datos
regs_despues_dane = len(df_traslados_consolidado)
match_municipios = df_traslados_consolidado['Codigo_Municipio'].notna().sum()
miss_municipios = regs_despues_dane - match_municipios

print("-" * 60)
print("📊 REPORTE DE CRUCE: CÓDIGO DANE MUNICIPIOS")
print("-" * 60)

if regs_antes_dane != regs_despues_dane:
    print(f"⚠️ ALERTA CRÍTICA: Violación de integridad geométrica en el DataFrame.")
    print(f"   Registros antes: {regs_antes_dane:,} | Después: {regs_despues_dane:,}")
else:
    print(f"✅ Integridad dimensional mantenida ({regs_antes_dane:,} registros).")

print(f"Códigos DANE mapeados: {match_municipios:,} ({(match_municipios/regs_antes_dane)*100:.1f}%)")
print(f"Municipios sin cruce:  {miss_municipios:,}")

# ============================================================================
# LIMPIEZA DE MEMORIA
# ============================================================================
# Eliminamos las variables temporales y el dataframe base según instrucción explícita
del df_mun_map, regs_antes_dane, regs_despues_dane, match_municipios, miss_municipios, df_municipios_sie
gc.collect()

## 5.3. Validar Correos
### 1. Explicación técnica del bloque
Se ha adaptado la lógica de validación de correos para el proceso de traslados:
- **Normalización de Dominio**: El código utiliza un diccionario de expresiones regulares para corregir errores comunes de digitación (ej: `@gamil` por `@gmail`).
- **Filtrado de "Placeholder"**: Implementa una doble validación para descartar correos de relleno como "na@...", "test@..." o "notiene@...", diferenciando entre coincidencias exactas en el nombre de usuario y patrones parciales.
- **Validación por Regex**: Solo los registros que cumplen estrictamente con el patrón `usuario@dominio.extension` sobreviven al proceso. Los fallidos se transforman en `None` (vacío).

### 2. Justificación de decisiones de optimización
- **Uso de `.apply()` controlado**: Aunque `.apply()` es menos eficiente que las funciones nativas de pandas, la lógica compleja de partición de usuario y chequeo de múltiples listas de exclusión hace que sea la solución más clara y robusta para este caso de limpieza.
- **Pre-procesamiento Vectorizado**: Las correcciones de dominio y limpieza de espacios se realizan de forma vectorizada antes de la validación estructural para maximizar la velocidad.

### 3. Validaciones de integridad aplicadas
- **Preservación de registros**: El proceso no elimina filas del DataFrame, solo vacía la celda de `correo_electronico` si no es válida, asegurando que no se pierda el rastro del afiliado para otros canales (como SMS).
- **Reporte de Efectividad**: Se incluye un cálculo porcentual de calidad de la fuente de datos SIE/BDUA, permitiendo auditar qué tan confiable es el canal de correo para este mes.

### 4. Listado de variables temporales eliminadas
- Se invoca `gc.collect()` para limpiar estructuras de datos generadas durante el mapeo de strings.

### 5. Posibles advertencias
- **Extensiones de dominio**: La expresión regular exige al menos 2 caracteres después del punto (ej: `.co`, `.com`). Correos mal terminados (ej: `usuario@gmail.c`) serán eliminados.

### 6. Validaciones sugeridas
- Ejecutar `df_traslados_consolidado['correo_electronico'].unique()` para confirmar visualmente que no quedaron términos como "na" o "sin correo".
- Revisar registros donde el correo sea `None` pero se tenga celular, para priorizar la notificación por SMS.

In [ ]:
# ============================================================================
# 5.3. LIMPIEZA Y VALIDACIÓN DE CORREOS ELECTRÓNICOS
# ============================================================================

def normalizar_y_validar_correos(df: pd.DataFrame, columna: str = 'correo_electronico') -> pd.DataFrame:
    """
    Normaliza correos, corrige dominios, elimina correos de relleno 
    y valida estructura mediante expresiones regulares.
    """
    
    # Diccionario de correcciones comunes en dominios
    correcciones_dominios = {
        r'@gamil\.': '@gmail.',
        r'@gamail\.': '@gmail.',
        r'@gimal\.': '@gmail.',
        r'@gimail\.': '@gmail.',
        r'@hotmial\.': '@hotmail.',
        r'@hotmal\.': '@hotmail.',
        r'@outlok\.': '@outlook.',
        r'@outluk\.': '@outlook.',
        r'@msn\.con$': '@msn.com',
        r'\.con$': '.com',
        r',com$': '.com',
    }

    # Palabras de relleno (Exactas y Parciales)
    palabras_exactas = ['na', 'test', 'prueba', 'correo', 'a', 'x']
    palabras_parciales = ['actualizar', 'notiene', 'sincorreo', 'noemail', 'noaplica']

    def validar_estructura(email: str) -> Optional[str]:
        # Manejo de nulos o vacíos
        if not email or email == 'nan' or email == '':
            return None
            
        parte_usuario = email.split('@')[0] if '@' in email else email
        
        # 1. Validación de palabras exactas en el nombre de usuario
        if parte_usuario in palabras_exactas:
            return None
            
        # 2. Validación de palabras parciales (contenido indeseado)
        if any(palabra in parte_usuario for palabra in palabras_parciales):
            return None

        # 3. Regex estándar para validación de formato email
        patron = r'^[a-z0-9._%+-]+@[a-z0-9.-]+\.[a-z]{2,}$'
        if re.match(patron, email):
            return email
            
        return None

    # --- Inicio del Proceso ---
    total_inicial = df[columna].dropna().replace('', None).count()
    
    # Normalización básica: minúsculas, sin espacios y corrección de comas
    df[columna] = df[columna].fillna('').astype(str).str.lower().str.strip()
    df[columna] = df[columna].str.replace(',', '.', regex=False)
    
    # Corrección de errores tipográficos en dominios
    for error, correccion in correcciones_dominios.items():
        df[columna] = df[columna].str.replace(error, correccion, regex=True)

    # Aplicación de la lógica de validación
    # Se genera una máscara para identificar correos reales
    correos_limpios = df[columna].apply(validar_estructura)
    mask_validos = correos_limpios.notna()
    
    validados = mask_validos.sum()
    eliminados = total_inicial - validados
    
    # Solo los correos que pasaron la validación permanecen, el resto se vuelve None (vacío)
    df[columna] = correos_limpios

    # --- Reporte de Calidad ---
    print("\n" + "="*60)
    print(f"{'REPORTE DE CALIDAD: CORREOS ELECTRÓNICOS':^60}")
    print("="*60)
    print(f"Registros iniciales con dato:        {total_inicial}")
    print(f"Correos que cumplen validación:      {validados}")
    print(f"Correos eliminados (basura/error):   {eliminados}")
    efectividad = (validados/total_inicial)*100 if total_inicial > 0 else 0
    print(f"Efectividad de la fuente:            {efectividad:.2f}%")
    print("-" * 60)
    
    return df

# Ejecución de la función sobre el consolidado enriquecido
df_traslados_consolidado = normalizar_y_validar_correos(df_traslados_consolidado)

# Limpieza de memoria (Principio 3)
gc.collect()

## 5.4. Validar Telefonos

### 1. Explicación técnica del bloque
Este bloque ejecuta un riguroso proceso de saneamiento de datos telefónicos para asegurar la efectividad de las campañas de SMS:
- **Normalización Numérica**: Utiliza expresiones regulares (`\D`) para eliminar paréntesis, guiones, puntos o espacios, dejando solo los dígitos puros.
- **Filtro de Entropía**: Aplica una validación lógica basada en el conteo de dígitos únicos (`set(numero)`). Esto descarta números falsos comunes en registros administrativos como "3000000000" o "3111111111".
- **Estandarización de Longitud**: Se enfoca exclusivamente en el formato de telefonía móvil colombiana (10 dígitos iniciando con 3).

### 2. Justificación de decisiones de optimización
- **Diferenciación de Nulos**: El reporte distingue entre registros que ya venían vacíos y aquellos que contenían basura. Esto permite medir la **Calidad del Dato** proveniente del SIE de manera aislada.
- **Manejo de Memoria**: La limpieza se realiza columna por columna para evitar la creación de múltiples copias del DataFrame principal, y se utiliza `mask_validos` (booleana) que es ligera en términos de consumo de bits.

### 3. Validaciones de integridad aplicadas
- **Check de Existencia**: La función verifica si la columna existe en el DataFrame antes de procesarla, evitando errores si el Maestro SIE cambia de estructura.
- **Preservación de registros**: Al igual que con los correos, no se eliminan filas, solo se limpian los campos individuales para que el afiliado no sea contactado con datos erróneos.
- **Validación de Tipo**: Se asegura que todos los datos procesados sean tratados como `str` antes de la limpieza para evitar errores con números científicos.

### 4. Listado de variables temporales eliminadas
- `columnas_telefonos`: Lista de configuración eliminada al final del bloque.
- Variables internas de la función (`reporte`, `mask_validos`, `stats`) son liberadas automáticamente al finalizar la ejecución del scope.

### 5. Posibles advertencias
- **Teléfonos Fijos**: La lógica actual está optimizada para **Celulares**. Si `telefono_1` o `telefono_2` contienen números fijos (7 dígitos), estos serán eliminados por no cumplir con la longitud de 10. Si se desea conservar fijos, la lógica de `es_valido` debe ajustarse.

### 6. Validaciones sugeridas
- Ejecutar `df_traslados_consolidado[df_traslados_consolidado['celular'].notna()]['celular'].str.len().unique()` para confirmar que solo existen números de 10 dígitos.
- Comparar el conteo de `NULOS FINAL` de la columna `celular` contra los resultados del bloque de correos para identificar cuántos usuarios quedaron totalmente incomunicados.

In [ ]:
# ============================================================================
# 5.4. LIMPIEZA Y VALIDACIÓN DE CONTACTOS TELEFÓNICOS
# ============================================================================

def limpiar_y_validar_telefonos(df: pd.DataFrame, columnas: List[str]) -> pd.DataFrame:
    """
    Limpia y valida números de teléfono (enfocado en celulares colombianos).
    
    Aplica reglas de:
    1. Limpieza de caracteres no numéricos.
    2. Validación estructural (10 dígitos, inicio con '3').
    3. Validación de calidad/entropía (mínimo 4 dígitos distintos para evitar 3000000000).
    """
    reporte: Dict[str, Dict[str, Any]] = {}

    def es_valido(numero: str) -> bool:
        # Si el dato es nulo o el string resultante de la limpieza es vacío
        if not numero or numero == 'nan' or numero == '':
            return False
        
        # Reglas de negocio para celular colombiano:
        # - Longitud exacta de 10 caracteres
        # - Debe iniciar con el prefijo '3'
        # - Entropía: debe tener al menos 4 dígitos diferentes para ser creíble
        return (len(numero) == 10 and 
                numero.startswith('3') and 
                len(set(numero)) >= 4)

    for col in columnas:
        if col not in df.columns:
            continue
            
        # 1. Conteo inicial de datos no nulos (para medir pérdida por calidad)
        # Se reemplazan strings vacíos por NaN para un conteo preciso de "datos reales"
        datos_reales_iniciales: int = df[col].replace('', None).dropna().count()
        
        # 2. Limpieza vectorizada: Eliminar cualquier caracter que no sea dígito
        df[col] = df[col].fillna('').astype(str).str.replace(r'\D', '', regex=True)
        
        # 3. Aplicación de validación de estructura y calidad
        mask_validos = df[col].apply(es_valido)
        
        # 4. Cálculo de métricas
        validados: int = mask_validos.sum()
        eliminados: int = datos_reales_iniciales - validados
        
        # 5. Estandarización final: lo que no es válido se convierte en None (vacío real)
        df.loc[~mask_validos, col] = None
        
        reporte[col] = {
            'iniciales': datos_reales_iniciales,
            'validados': validados,
            'eliminados': eliminados,
            'nulos_final': df[col].isna().sum()
        }

    # --- Salida de Auditoría Técnica ---
    print("\n" + "="*85)
    print(f"{'AUDITORÍA DE CALIDAD TELEFÓNICA':^85}")
    print("="*85)
    print(f"{'COLUMNA':<20} | {'EXISTENTES':<12} | {'VALIDADOS':<12} | {'ELIMINADOS':<12} | {'NULOS FINAL'}")
    print("-" * 85)
    
    for col, stats in reporte.items():
        print(f"{col:<20} | {stats['iniciales']:<12} | {stats['validados']:<12} | {stats['eliminados']:<12} | {stats['nulos_final']}")
    print("="*85 + "\n")
    
    return df

# Ejecución del proceso sobre las columnas de contacto
columnas_telefonos: List[str] = ['celular', 'telefono_1', 'telefono_2']
df_traslados_consolidado = limpiar_y_validar_telefonos(df_traslados_consolidado, columnas_telefonos)

# Limpieza de variables temporales (Principio 3)
del columnas_telefonos
gc.collect()

## 5.5. Asunto y radicado

### 1. Explicación técnica del bloque
Este bloque prepara los campos finales requeridos para las herramientas de notificación masiva:
- **Columna RADICADO**: Se inicializa como `None` (vacía) para todos los registros, cumpliendo con el requerimiento de estructura para futuros cruces o llenados manuales.
- **Columna ASUNTO Dinámica**: Implementa una fórmula de concatenación que incluye el periodo de proceso (traducido a nombre de mes), la EPS receptora y la identificación completa del afiliado.
- **Condicionalidad**: El asunto solo se construye para filas donde `correo_electronico` no sea nulo, optimizando el tamaño del archivo final y evitando procesar registros que solo se notificarán por SMS.

### 2. Justificación de decisiones de optimización
- **Vectorización de Strings**: Se utiliza la suma de series de pandas (`+`) en lugar de `.apply(lambda x: ...)` o bucles `for`. Esto permite que la construcción del texto se realice a nivel de CPU de forma paralela en todos los registros válidos.
- **Manejo de Placeholders**: Se aplica `.fillna("OTRA ENTIDAD")` temporalmente durante la creación del asunto para asegurar que, si una EPS no fue encontrada en el maestro, el mensaje siga siendo legible.

### 3. Validaciones de integridad aplicadas
- **Principio de Idempotencia**: Las columnas se definen explícitamente al inicio. Si se vuelve a ejecutar la celda, los valores se resetean correctamente sin duplicar información.
- **Sincronización con Limpieza Previa**: La lógica depende directamente de la máscara de `correo_electronico` generada en el bloque de validación anterior, garantizando que solo los correos que pasaron el filtro de calidad reciban un asunto.

### 4. Listado de variables temporales eliminadas
- `meses_nombres`, `mes_texto`: Diccionario y string de apoyo para fechas.
- `mask_correo_valido`: Máscara booleana de selección.
- `eps_destino`: Serie temporal de nombres de EPS.
- `conteo_radicados`: Variable de control de integridad.

### 5. Posibles advertencias
- **Longitud del Asunto**: Algunos clientes de correo pueden truncar asuntos extremadamente largos. La combinación de nombres de EPS largos e identificaciones extensas podría generar asuntos de más de 100 caracteres.

### 6. Validaciones sugeridas
- Ejecutar `df_traslados_consolidado[df_traslados_consolidado['ASUNTO'].notna()].head(1)` para verificar visualmente que la redacción del asunto cumple con el formato solicitado.
- Confirmar que la cantidad de filas con `ASUNTO` sea exactamente igual a la cantidad de `validados` reportada en el bloque de limpieza de correos electrónicos.

In [ ]:
# ============================================================================
# 5.5. GENERACIÓN DE COLUMNAS PARA NOTIFICACIÓN (ASUNTO Y RADICADO)
# ============================================================================

# 1. Definición de nombres de meses en español para el asunto
meses_nombres: Dict[int, str] = {
    1: "Enero", 2: "Febrero", 3: "Marzo", 4: "Abril",
    5: "Mayo", 6: "Junio", 7: "Julio", 8: "Agosto",
    9: "Septiembre", 10: "Octubre", 11: "Noviembre", 12: "Diciembre"
}

mes_texto: str = meses_nombres.get(MES_PROCESO, "Desconocido")

# 2. Inicialización de columnas (Principio 2: Reproducibilidad)
# ASUNTO inicialmente vacío (None)
df_traslados_consolidado['ASUNTO'] = None

# RADICADO siempre vacío para este proceso
df_traslados_consolidado['RADICADO'] = None

# 3. Construcción vectorizada del ASUNTO para registros con correo válido
# Máscara de registros que tienen correo electrónico tras la limpieza previa
mask_correo_valido = df_traslados_consolidado['correo_electronico'].notna()

if mask_correo_valido.any():
    # Construcción de la cadena dinámica (Principio 3: Operación vectorizada)
    # Se maneja el caso de NOMBRE_EPS_DESTINO nulo con un valor por defecto para evitar 'nan' en el texto
    eps_destino = df_traslados_consolidado['NOMBRE_EPS_DESTINO'].fillna("OTRA ENTIDAD")
    
    df_traslados_consolidado.loc[mask_correo_valido, 'ASUNTO'] = (
        f"Notificación traslados de salida aprobado {mes_texto}, {AÑO_PROCESO} para " + 
        eps_destino + ", " + 
        df_traslados_consolidado['TPS_IDN_ID'] + " " + 
        df_traslados_consolidado['HST_IDN_NUMERO_IDENTIFICACION']
    )
    
    print(f"✅ Se generaron {mask_correo_valido.sum():,} asuntos personalizados para envío de correos.")
else:
    print("⚠️ Aviso: No se detectaron correos válidos para generar asuntos.")

# 4. Validación de integridad: Confirmar que RADICADO sea 100% nulo
conteo_radicados = df_traslados_consolidado['RADICADO'].notna().sum()
if conteo_radicados > 0:
    print(f"⚠️ Alerta: Se detectaron {conteo_radicados} radicados no vacíos inesperadamente.")

# ============================================================================
# LIMPIEZA DE MEMORIA
# ============================================================================
del meses_nombres, mes_texto, mask_correo_valido, conteo_radicados
if 'eps_destino' in locals(): del eps_destino
gc.collect()

# 6. Guardar excel

### 1. Explicación técnica del bloque
Este bloque finaliza el flujo de trabajo generando el entregable en formato Excel (.xlsx):
- **Estructura Multi-hoja**: Utiliza `pd.ExcelWriter` para empaquetar diferentes vistas de los datos en un solo archivo, lo cual facilita la distribución a diferentes áreas (Ventanilla Única vs. Aseguramiento).
- **Segmentación de Ventanilla**: Se realiza un filtrado final sobre `ASUNTO` para extraer únicamente a los usuarios listos para ser notificados vía correo electrónico, incluyendo las columnas de identificación necesarias para que el área administrativa asigne los números de radicado.
- **Hoja de Respaldo**: La hoja `Consolidado_Salidas` preserva la integridad de toda la operación, incluyendo afiliados con y sin datos de contacto, para auditorías posteriores.

### 2. Justificación de decisiones de optimización
- **Context Manager (`with`)**: El uso de `with pd.ExcelWriter` asegura que el archivo se cierre y se libere de la memoria RAM correctamente, incluso si ocurre un error durante el proceso de escritura.
- **Engine `openpyxl`**: Se especifica este motor por ser el estándar más robusto para la creación de archivos Excel modernos, permitiendo el manejo de múltiples hojas sin corromper los metadatos.

### 3. Validaciones de integridad aplicadas
- **Check de Registros con Asunto**: Al filtrar por `ASUNTO.notna()`, garantizamos que la hoja 'Ventanilla' no contenga filas inútiles (sin correo), optimizando el trabajo del área receptora.
- **Persistencia de Datos**: Se utiliza `.copy()` al crear `df_ventanilla` para evitar el riesgo de modificar accidentalmente el DataFrame maestro durante la exportación.

### 4. Listado de variables temporales eliminadas
- `df_ventanilla`: Vista parcial de los datos.
- `columnas_ventanilla`: Lista de selección de campos.
- `nombre_archivo` y `ruta_final_excel`: Strings de configuración de ruta.

### 5. Posibles advertencias
- **Bloqueo de Archivo**: Si el archivo Excel ya existe y está abierto por otro usuario (o por ti mismo), el script fallará con un error de "Permiso denegado". Asegúrate de cerrarlo antes de re-ejecutar la celda.
- **Límite de Filas**: Excel soporta hasta 1,048,576 filas. Si el consolidado supera este límite, el guardado fallará (poco probable para traslados mensuales).

### 6. Validaciones sugeridas
- Abrir el archivo generado y verificar que la columna `RADICADO` en la hoja 'Ventanilla' esté efectivamente vacía y lista para ser diligenciada.
- Confirmar que la suma de registros en la hoja 'Consolidado_Salidas' coincida con el total reportado tras la limpieza de duplicados de `AFL_ID`.

In [ ]:
# ============================================================================
# 6. EXPORTACIÓN DE RESULTADOS (EXCEL MULTI-HOJA)
# ============================================================================

# 1. Definición del nombre del archivo final
nombre_archivo: str = f"Notificaciones_Traslados_Salida_{MES_PROCESO:02d}_{AÑO_PROCESO}.xlsx"
ruta_final_excel: Path = R_SALIDA_BASE / nombre_archivo

# 2. Preparación de datos para la hoja 'Ventanilla'
# Solo registros que tienen un ASUNTO generado (implica que tienen correo válido)
columnas_ventanilla: list = [
    'AFL_ID', 
    'TPS_IDN_ID', 
    'HST_IDN_NUMERO_IDENTIFICACION', 
    'correo_electronico', 
    'ASUNTO', 
    'RADICADO'
]

df_ventanilla = df_traslados_consolidado[
    df_traslados_consolidado['ASUNTO'].notna()
][columnas_ventanilla].copy()

# 3. Proceso de guardado con ExcelWriter (Optimización de recursos)
try:
    with pd.ExcelWriter(ruta_final_excel, engine='openpyxl') as writer:
        # Hoja 1: Insumo para radicación masiva
        df_ventanilla.to_excel(writer, sheet_name='Ventanilla', index=False)
        
        # Hoja 2: Base completa para trazabilidad y auditoría
        df_traslados_consolidado.to_excel(writer, sheet_name='Consolidado_Salidas', index=False)
    
    print(f"✅ Archivo guardado exitosamente en: {ruta_final_excel}")
    print(f"📊 Resumen de hojas:")
    print(f"   - Hoja 'Ventanilla': {len(df_ventanilla):,} registros (con correo).")
    print(f"   - Hoja 'Consolidado_Salidas': {len(df_traslados_consolidado):,} registros (total).")

except Exception as e:
    print(f"❌ Error al guardar el archivo: {e}")

# ============================================================================
# LIMPIEZA FINAL DE MEMORIA
# ============================================================================
del df_ventanilla, columnas_ventanilla, nombre_archivo, ruta_final_excel
gc.collect()